# Notebook 05 — Pixel-Level SPI × Geotagged News Spatial Corroboration

**New in this notebook vs NB04:**
| NB04 | NB05 |
|------|------|
| Country-level aggregation | ERA5 pixel level (~28km grid) |
| Query by country name | Query by bounding box + reverse-geocoded place name |
| No spatial matching | cKDTree spatial match: article location ↔ SPI pixel |
| Google News only | GDELT DOC 2.0 (65K sources) |
| No ground truth | HDX/FEWS NET IPC food security labels |

**Pipeline**:
```
323K SPI pixels
  → filter Severe/Extreme → top-100 worst drought_ids
  → 1° grid cluster → ~200-800 regions
  → reverse geocode centroid (Nominatim)
  → GDELT query per region (date + place)
  → spaCy NER → Nominatim → lat/lon per article
  → cKDTree spatial match (1° buffer, ±30d temporal)
  → spatial_corroboration_score per region
  → weighted aggregate per drought_id
  → spatial heatmap
```

## Step 1 — Install

In [ ]:
import sys, subprocess

# typing_extensions must be upgraded FIRST — spaCy requires >=4.10 for Sentinel
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','typing_extensions'], check=True)

pkgs = ['scikit-learn','shapely','geopandas','geopy','spacy',
        'requests','feedparser','pandas','numpy','matplotlib','seaborn']
for p in pkgs:
    subprocess.run([sys.executable,'-m','pip','install','-q',p], capture_output=True)
# spaCy English model
subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm','-q'], capture_output=True)
print("Dependencies ready.")
print("IMPORTANT: restart kernel now, then rerun from Step 1.")

## Step 2 — Imports & Config

In [ ]:
import os, ssl, json, re, time, warnings, urllib.request
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
from scipy.spatial import cKDTree
import spacy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
PIXEL_CSV = os.path.expanduser(
    "~/Library/CloudStorage/Box-Box/R - Projects - Tharaka/"
    "Drought-indices/Data/all_drought_events_pixelbypixel.csv"
)

# ── Config ───────────────────────────────────────────────────────────────
TOP_N_EVENTS      = 100    # worst drought_ids to process
GRID_CELL_DEG     = 1.0    # cluster ERA5 pixels into 1° grid cells
MIN_PIXELS_REGION = 4      # drop tiny isolated clusters
SPATIAL_BUFFER    = 1.0    # degrees (~111km) for article↔pixel matching
TEMPORAL_LAG_DAYS = 30     # extra days after event end for late reporting
MAX_GDELT_RECORDS = 50     # articles per GDELT query

SEVERITY_KEEP = ['Severe','Extreme']   # filter out Mild/Moderate pixels

# ── SSL context ─────────────────────────────────────────────────────────
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode   = ssl.CERT_NONE

HEADERS = {'User-Agent':'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Chrome/120.0.0.0'}

print("Config ready.")
print(f"Pixel CSV: {os.path.exists(PIXEL_CSV)}")

## Step 3 — Load Pixel CSV & Filter

`all_drought_events_pixelbypixel.csv` has 323,000 rows — one per pixel per drought event.

Columns: `drought_id, start_date, end_date, x (lon), y (lat), severity_class, peak_spi, spi_scale`

Filter strategy:
1. Keep only **Severe/Extreme** pixels (removes ~60% of rows)
2. Keep only **SPI3** scale
3. Take worst **100 drought_ids** by peak_spi — this is our computational ceiling

In [ ]:
try:
    df_pix = pd.read_csv(PIXEL_CSV, parse_dates=['start_date','end_date','peak_date'])
    print(f"Loaded: {len(df_pix):,} rows")
    print(f"Columns: {list(df_pix.columns)}")
except FileNotFoundError:
    print("SPI file not found — using synthetic sample pixels")
    # Generate a synthetic 10x10 pixel grid around Ethiopia / Kenya region
    lons  = np.arange(36, 46, 0.25)
    lats  = np.arange(4, 14, 0.25)
    rows  = []
    for i, (lo, la) in enumerate([(x,y) for x in lons for y in lats]):
        rows.append({
            'drought_id': 1 if la > 9 else 2,
            'start_date': pd.Timestamp('2022-03-01'),
            'end_date':   pd.Timestamp('2022-07-01'),
            'peak_date':  pd.Timestamp('2022-05-01'),
            'duration_months': 4,
            'peak_spi':  np.random.uniform(-2.5, -1.3),
            'total_severity': np.random.uniform(-10, -4),
            'severity_class': np.random.choice(['Severe','Extreme'], p=[0.6,0.4]),
            'x': lo, 'y': la,
            'spi_scale': 'SPI3',
        })
    df_pix = pd.DataFrame(rows)
    print(f"Synthetic data: {len(df_pix)} pixels")

# ── Filter: severity ─────────────────────────────────────────────────────
df_f = df_pix[df_pix['severity_class'].isin(SEVERITY_KEEP)].copy()
print(f"After severity filter: {len(df_f):,} rows ({len(df_f)/len(df_pix)*100:.0f}%)")

# ── Filter: SPI3 ─────────────────────────────────────────────────────────
if 'spi_scale' in df_f.columns:
    df_f = df_f[df_f['spi_scale'] == 'SPI3']
    print(f"After SPI3 filter: {len(df_f):,} rows")

# ── Top-N worst drought_ids ───────────────────────────────────────────────
worst_ids = (
    df_f.groupby('drought_id')['peak_spi'].min()
    .sort_values().head(TOP_N_EVENTS).index
)
df_work = df_f[df_f['drought_id'].isin(worst_ids)].copy()
print(f"Top-{TOP_N_EVENTS} drought_ids: {len(df_work):,} pixels")
print(f"Lat range: {df_work['y'].min():.1f}° to {df_work['y'].max():.1f}°")
print(f"Lon range: {df_work['x'].min():.1f}° to {df_work['x'].max():.1f}°")
print(f"Date range: {df_work['start_date'].min().date()} to {df_work['end_date'].max().date()}")

## Step 4 — Grid Cluster Pixels into Regions

**Why not process 60K pixels individually?**
60K pixels × 1 GDELT query each = thousands of API calls, hours of runtime.

**Solution**: aggregate into 1° grid cells.
Each 1° cell = ~111km × ~111km region. 
This collapses 60K pixels to ~200-800 scraping jobs.

**Why 1° grid, not DBSCAN?**
DBSCAN on 40K points requires ~12GB distance matrix. 
Grid is O(n), memory-safe, produces axis-aligned bounding boxes for GDELT queries.

In [ ]:
def cluster_pixels(df_event, grid_deg=GRID_CELL_DEG, min_px=MIN_PIXELS_REGION):
    df = df_event.copy()
    df['grid_x'] = (df['x'] / grid_deg).round(0) * grid_deg
    df['grid_y'] = (df['y'] / grid_deg).round(0) * grid_deg

    regions = df.groupby(['drought_id','grid_x','grid_y']).agg(
        centroid_lon  = ('x', 'mean'),
        centroid_lat  = ('y', 'mean'),
        lon_min       = ('x', 'min'),
        lon_max       = ('x', 'max'),
        lat_min       = ('y', 'min'),
        lat_max       = ('y', 'max'),
        n_pixels      = ('x', 'count'),
        mean_spi      = ('peak_spi', 'mean'),
        min_spi       = ('peak_spi', 'min'),
        start_date    = ('start_date', 'min'),
        end_date      = ('end_date', 'max'),
        severity_mode = ('severity_class', lambda x: x.mode().iloc[0]),
    ).reset_index()

    # Drop micro-clusters (noise)
    regions = regions[regions['n_pixels'] >= min_px].reset_index(drop=True)
    regions['region_id'] = [f"R{i:04d}" for i in range(len(regions))]
    return regions

df_regions = cluster_pixels(df_work)
print(f"Regions after clustering: {len(df_regions)}")
print(f"Grid cell size: {GRID_CELL_DEG}°  (~{GRID_CELL_DEG*111:.0f}km)")
print()
print("Top 10 largest regions:")
print(df_regions.nlargest(10,'n_pixels')[
    ['region_id','centroid_lon','centroid_lat','n_pixels','min_spi','start_date','end_date']
].to_string(index=False))

## Step 5 — Visualise SPI Regions

Plot pixel centroids coloured by SPI severity + cluster region boundaries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: pixel scatter (SPI intensity)
sc = axes[0].scatter(df_work['x'], df_work['y'],
                     c=df_work['peak_spi'], cmap='RdBu',
                     vmin=-3, vmax=0, s=1, alpha=0.4)
plt.colorbar(sc, ax=axes[0], label='Peak SPI')
axes[0].set_title(f'SPI Pixel Map ({len(df_work):,} pixels)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Right: region centroids coloured by cluster intensity
sc2 = axes[1].scatter(df_regions['centroid_lon'], df_regions['centroid_lat'],
                      c=df_regions['min_spi'], cmap='RdBu',
                      vmin=-3, vmax=0,
                      s=df_regions['n_pixels']*0.5 + 10,
                      alpha=0.7, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc2, ax=axes[1], label='Min SPI in region')
axes[1].set_title(f'Clustered Regions ({len(df_regions)} regions, {GRID_CELL_DEG}° grid)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.savefig('nb05_regions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: nb05_regions.png")

## Step 6 — Reverse Geocode Region Centroids

**Why**: GDELT text search needs place names (e.g. "drought Ethiopia Ogaden"), not lat/lon. Nominatim reverse-geocodes each region centroid to a place name.

Nominatim = free OpenStreetMap geocoding service. Rate limit: 1 req/sec.

In [ ]:
geolocator = Nominatim(user_agent='drought-spi-research-v1', timeout=10)
geocode_cache = {}

def reverse_geocode(lat, lon):
    key = (round(lat,1), round(lon,1))
    if key in geocode_cache:
        return geocode_cache[key]
    try:
        loc = geolocator.reverse((lat, lon), language='en', zoom=6)
        if loc and loc.raw:
            addr = loc.raw.get('address',{})
            place = (addr.get('state') or addr.get('province') or
                     addr.get('county') or addr.get('country') or '')
            country = addr.get('country', '')
            result = {'place': place, 'country': country}
        else:
            result = {'place': '', 'country': ''}
    except Exception:
        result = {'place': '', 'country': ''}
    geocode_cache[key] = result
    time.sleep(1.0)   # Nominatim rate limit: 1 req/sec
    return result

print(f"Reverse geocoding {len(df_regions)} region centroids...")
print("(~1 second per region — expected runtime: "
      f"{len(df_regions)} seconds = {len(df_regions)//60} min {len(df_regions)%60} sec)")

places = []
for _, row in df_regions.iterrows():
    geo = reverse_geocode(row['centroid_lat'], row['centroid_lon'])
    places.append(geo)

df_regions['place_name'] = [p['place'] for p in places]
df_regions['country']    = [p['country'] for p in places]

print("\nSample geocoded regions:")
print(df_regions[['region_id','centroid_lon','centroid_lat','place_name','country','min_spi']
                 ].head(10).to_string(index=False))

## Step 7 — GDELT DOC 2.0 Query per Region

**GDELT DOC 2.0** indexes 65,000+ news sources globally, updated every 15 min.
No API key needed. Max 250 records per call.

Query = `drought {place_name} {country}` within the event date window.

In [ ]:
def gdelt_query_region(region):
    place  = region.get('place_name','')
    country= region.get('country','')
    q_parts= ['drought']
    if place:   q_parts.append(place)
    if country and country != place: q_parts.append(country)
    query  = ' '.join(q_parts)

    t_start = region['start_date']
    t_end   = region['end_date'] + pd.Timedelta(days=TEMPORAL_LAG_DAYS)

    # GDELT max window: clip to 1 year
    if (t_end - t_start).days > 365:
        t_end = t_start + pd.Timedelta(days=365)

    params = {
        'query':         query,
        'mode':          'artlist',
        'maxrecords':    MAX_GDELT_RECORDS,
        'startdatetime': t_start.strftime('%Y%m%d%H%M%S'),
        'enddatetime':   t_end.strftime('%Y%m%d%H%M%S'),
        'format':        'json',
    }
    try:
        r = requests.get('https://api.gdeltproject.org/api/v2/doc/doc',
                         params=params, timeout=25)
        arts = r.json().get('articles', [])
        return [{
            'region_id':  region['region_id'],
            'title':      a.get('title',''),
            'url':        a.get('url',''),
            'source':     a.get('domain',''),
            'language':   a.get('language','English'),
            'published':  a.get('seendate',''),
            'source_country': a.get('sourcecountry',''),
            '_query':     query,
        } for a in arts]
    except Exception as e:
        print(f"    GDELT error ({region['region_id']}): {e}")
        return []

# ── Run for all regions ───────────────────────────────────────────────────
print(f"Querying GDELT for {len(df_regions)} regions...")
print("(~0.5 sec per region)")

all_articles = []
for idx, row in df_regions.iterrows():
    arts = gdelt_query_region(row)
    all_articles.extend(arts)
    if (idx+1) % 20 == 0:
        print(f"  [{idx+1}/{len(df_regions)}] Total articles so far: {len(all_articles)}")
    time.sleep(0.5)

df_arts = pd.DataFrame(all_articles)
if not df_arts.empty:
    df_arts = df_arts.drop_duplicates('url').reset_index(drop=True)
print(f"\nTotal unique articles fetched: {len(df_arts)}")
if not df_arts.empty:
    print(f"Languages: {df_arts['language'].value_counts().head(8).to_dict()}")
    print(f"Source countries: {df_arts['source_country'].value_counts().head(8).to_dict()}")

## Step 7b — Sample Fallback (if GDELT returns 0 articles)

If network blocked or GDELT returns nothing, inject synthetic articles for testing.

In [ ]:
if df_arts.empty or len(df_arts) < 5:
    print("GDELT returned few/no articles — injecting synthetic sample for testing")

    sample_regions = df_regions.head(5)
    synthetic = []
    for _, reg in sample_regions.iterrows():
        place = reg['place_name'] or reg['country'] or 'the region'
        synthetic += [
            {'region_id': reg['region_id'],
             'title': f'Severe drought grips {place}, crops fail across region',
             'url': f'sample-{reg["region_id"]}-1',
             'source': 'SampleNews', 'language': 'English',
             'published': reg['start_date'].strftime('%Y%m%dT%H%M%SZ'),
             'source_country': '', '_query': f'drought {place}'},
            {'region_id': reg['region_id'],
             'title': f'Water shortage crisis in {place}, thousands affected',
             'url': f'sample-{reg["region_id"]}-2',
             'source': 'SampleHumanitarian', 'language': 'English',
             'published': reg['start_date'].strftime('%Y%m%dT%H%M%SZ'),
             'source_country': '', '_query': f'drought {place}'},
            {'region_id': reg['region_id'],
             'title': f'Football results: weekend league matches',
             'url': f'sample-{reg["region_id"]}-3',
             'source': 'SampleSports', 'language': 'English',
             'published': reg['start_date'].strftime('%Y%m%dT%H%M%SZ'),
             'source_country': '', '_query': f'drought {place}'},
        ]
    df_arts = pd.DataFrame(synthetic)
    print(f"Synthetic articles: {len(df_arts)}")

## Step 8 — Drought Filter + Geoparse Articles

**Two tasks per article:**
1. **Is it about drought?** — keyword filter (fast, no model needed for this step)
2. **Where?** — spaCy NER extracts place names → Nominatim → lat/lon

**Why spaCy here and not mordecai3?**
mordecai3 requires ~500MB GeoNames database download + setup. 
spaCy `en_core_web_sm` + Nominatim is lighter and works on MacBook CPU.
mordecai3 gives better accuracy for ambiguous African place names — use it in production.

In [ ]:
# ── Load spaCy ───────────────────────────────────────────────────────────
nlp = spacy.load('en_core_web_sm')

DROUGHT_KW = ['drought','dry spell','water shortage','rainfall deficit',
              'crop failure','famine','water scarcity','precipitation deficit',
              'arid','food insecurity','livestock deaths','pastoralist']

def is_drought_article(title, text=''):
    combined = (title + ' ' + text).lower()
    return any(kw in combined for kw in DROUGHT_KW)

fwd_geocode_cache = {}

def geocode_place(place_name):
    if not place_name or place_name in fwd_geocode_cache:
        return fwd_geocode_cache.get(place_name)
    try:
        loc = geolocator.geocode(place_name, language='en', timeout=8)
        if loc:
            result = (loc.latitude, loc.longitude)
        else:
            result = None
    except Exception:
        result = None
    fwd_geocode_cache[place_name] = result
    time.sleep(1.0)
    return result

def geoparse_article(title, max_places=2):
    """Extract top-2 place names from title → geocode → lat/lon list."""
    doc = nlp(title[:512])
    places = [ent.text for ent in doc.ents if ent.label_ in ('GPE','LOC')]
    coords = []
    for place in list(dict.fromkeys(places))[:max_places]:  # dedup, keep order
        loc = geocode_place(place)
        if loc:
            coords.append({'place': place, 'lat': loc[0], 'lon': loc[1]})
    return coords

# ── Process articles ──────────────────────────────────────────────────────
print("Processing articles: drought filter + geoparse...")
rows = []
n_total = len(df_arts)

for i, (_, art) in enumerate(df_arts.iterrows()):
    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{n_total}]")

    # Drought filter
    if not is_drought_article(art.get('title','')):
        continue

    # Geoparse
    coords = geoparse_article(art.get('title',''))

    if coords:
        for c in coords:
            rows.append({**art.to_dict(), 'art_lat': c['lat'],
                         'art_lon': c['lon'], 'art_place': c['place']})
    else:
        # No location extracted — keep article without coords (country-level fallback)
        rows.append({**art.to_dict(), 'art_lat': None,
                     'art_lon': None, 'art_place': None})

df_geo = pd.DataFrame(rows)
df_with_coords = df_geo[df_geo['art_lat'].notna()]

print(f"\nDrought-relevant articles: {len(df_geo)}")
print(f"  With lat/lon: {len(df_with_coords)}")
print(f"  Without coords: {len(df_geo)-len(df_with_coords)}")
print(f"  Geocoded places: {df_with_coords['art_place'].value_counts().head(10).to_dict()}")

## Step 9 — Spatial Matching: Articles ↔ SPI Regions

**cKDTree** = fast nearest-neighbour search.
Build tree from region centroids → for each article, find all regions within 1° radius.

**1° buffer** (~111km): covers ERA5 1° cluster + news geoparsing ~50km error.
**Temporal check**: article `published` must be within `[start_date, end_date + 30d]`.

In [ ]:
# ── Build KDTree from region centroids ───────────────────────────────────
region_coords = df_regions[['centroid_lon','centroid_lat']].values
tree = cKDTree(region_coords)

# ── Parse article dates ───────────────────────────────────────────────────
def parse_date(s):
    if not s: return None
    for fmt in ('%Y%m%dT%H%M%SZ','%Y%m%d%H%M%S','%a, %d %b %Y %H:%M:%S %z'):
        try: return pd.Timestamp(datetime.strptime(str(s)[:len(fmt)], fmt))
        except: pass
    try: return pd.Timestamp(str(s)[:10])
    except: return None

df_with_coords['pub_ts'] = df_with_coords['published'].apply(parse_date)

# ── Match ─────────────────────────────────────────────────────────────────
matched = []
for _, art in df_with_coords.iterrows():
    art_pt = np.array([art['art_lon'], art['art_lat']])
    idxs = tree.query_ball_point(art_pt, r=SPATIAL_BUFFER)
    for idx in idxs:
        reg = df_regions.iloc[idx]
        # Temporal check
        t_start = reg['start_date'] - pd.Timedelta(days=7)
        t_end   = reg['end_date']   + pd.Timedelta(days=TEMPORAL_LAG_DAYS)
        pub_ts  = art['pub_ts']
        if pub_ts is None or (t_start <= pub_ts <= t_end):
            matched.append({
                'region_id':   reg['region_id'],
                'drought_id':  reg['drought_id'],
                'title':       art['title'],
                'art_place':   art['art_place'],
                'art_lat':     art['art_lat'],
                'art_lon':     art['art_lon'],
                'published':   art['published'],
                'source':      art['source'],
                'language':    art['language'],
                'spatial_dist': float(np.sqrt(
                    (art['art_lon']-reg['centroid_lon'])**2 +
                    (art['art_lat']-reg['centroid_lat'])**2
                )),
            })

df_matched = pd.DataFrame(matched)
print(f"Spatially matched articles: {len(df_matched)}")
print(f"Regions with ≥1 match: {df_matched['region_id'].nunique()} / {len(df_regions)}")
if not df_matched.empty:
    print(f"\nTop matched regions:")
    print(df_matched.groupby('region_id').size().sort_values(ascending=False).head(10))

## Step 10 — Compute Spatial Corroboration Scores

Two levels:
1. **Region level**: n_matched_articles / n_total_articles_queried
2. **Event level**: weighted average of region scores (weight = n_pixels in region)

In [ ]:
# ── Merge: how many articles queried per region ───────────────────────────
n_queried = df_arts.groupby('region_id').size().rename('n_queried')
n_matched = df_matched.groupby('region_id').size().rename('n_matched') if not df_matched.empty else pd.Series(dtype=int, name='n_matched')

df_regions = df_regions.set_index('region_id')
df_regions = df_regions.join(n_queried).join(n_matched)
df_regions['n_queried'] = df_regions['n_queried'].fillna(0).astype(int)
df_regions['n_matched'] = df_regions['n_matched'].fillna(0).astype(int)
df_regions['spatial_score'] = (
    df_regions['n_matched'] / df_regions['n_queried'].clip(lower=1)
).round(3)
df_regions = df_regions.reset_index()

# ── Event-level weighted aggregate ───────────────────────────────────────
def weighted_mean(group):
    w = group['n_pixels'].values
    s = group['spatial_score'].values
    return np.average(s, weights=w) if w.sum() > 0 else 0.0

df_events = df_regions.groupby('drought_id').apply(weighted_mean).rename('spatial_score').reset_index()
df_events_meta = df_regions.groupby('drought_id').agg(
    total_pixels      = ('n_pixels','sum'),
    n_regions         = ('region_id','count'),
    min_spi           = ('min_spi','min'),
    n_articles_total  = ('n_queried','sum'),
    n_articles_matched= ('n_matched','sum'),
    countries         = ('country', lambda x: ', '.join(x.dropna().unique()[:3])),
    start_date        = ('start_date','min'),
    end_date          = ('end_date','max'),
).reset_index()
df_events = df_events.merge(df_events_meta, on='drought_id')
df_events = df_events.sort_values('spatial_score', ascending=False)

print("Event-level spatial corroboration scores:")
print(df_events[['drought_id','countries','min_spi','total_pixels',
                 'n_articles_total','n_articles_matched','spatial_score']
               ].to_string(index=False))

## Step 11 — Spatial Heatmap

**Two-layer map:**
- Layer 1: SPI pixel scatter (blue=dry, red=normal)
- Layer 2: Region bubbles sized by n_matched_articles, coloured by spatial_score

Events with both dark SPI colour AND large bubble = **high confidence drought with news evidence**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Pixel-Level SPI × Geotagged News Spatial Corroboration', fontsize=14, fontweight='bold')

# ── Try geopandas world basemap ───────────────────────────────────────────
try:
    import geopandas as gpd
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    for ax in axes:
        world.plot(ax=ax, color='#f5f5f5', edgecolor='#999999', linewidth=0.4)
    print("World basemap loaded.")
except Exception as e:
    print(f"geopandas basemap failed ({e}) — plain scatter plot")

# ── Left: SPI pixel scatter ───────────────────────────────────────────────
sc1 = axes[0].scatter(df_work['x'], df_work['y'],
                      c=df_work['peak_spi'], cmap='RdBu_r',
                      vmin=-3, vmax=0, s=2, alpha=0.35)
plt.colorbar(sc1, ax=axes[0], label='Peak SPI')
axes[0].set_title(f'SPI Pixels: Severe/Extreme Events
({len(df_work):,} pixels, {GRID_CELL_DEG}° clusters)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Overlay region bounding boxes
for _, reg in df_regions.iterrows():
    from matplotlib.patches import Rectangle
    rect = Rectangle((reg['lon_min'], reg['lat_min']),
                      reg['lon_max']-reg['lon_min'],
                      reg['lat_max']-reg['lat_min'],
                      linewidth=0.5, edgecolor='gray', facecolor='none', alpha=0.4)
    axes[0].add_patch(rect)

# ── Right: news density bubbles over SPI ─────────────────────────────────
sc2 = axes[1].scatter(df_work['x'], df_work['y'],
                      c=df_work['peak_spi'], cmap='RdBu_r',
                      vmin=-3, vmax=0, s=1, alpha=0.2)

# Bubbles: only regions with queried articles
reg_with_arts = df_regions[df_regions['n_queried'] > 0]
if len(reg_with_arts):
    sc3 = axes[1].scatter(
        reg_with_arts['centroid_lon'],
        reg_with_arts['centroid_lat'],
        s = np.clip(reg_with_arts['n_matched'] * 20 + 30, 30, 500),
        c = reg_with_arts['spatial_score'],
        cmap='YlOrRd', vmin=0, vmax=1,
        edgecolors='black', linewidths=0.5, alpha=0.85, zorder=5
    )
    plt.colorbar(sc3, ax=axes[1], label='Spatial Corroboration Score')

axes[1].set_title('News Density per Region
(bubble size = matched articles, colour = score)')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')

# Focus on data extent
pad = 5
for ax in axes:
    ax.set_xlim(df_work['x'].min()-pad, df_work['x'].max()+pad)
    ax.set_ylim(df_work['y'].min()-pad, df_work['y'].max()+pad)

plt.tight_layout()
plt.savefig('nb05_spatial_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: nb05_spatial_heatmap.png")

## Step 12 — Ground Truth Validation Layer: HDX / FEWS NET

**What is FEWS NET?** Famine Early Warning Systems Network — USAID-funded.
Publishes IPC (Integrated Food Security Phase Classification) maps:
- Phase 3 = Crisis, Phase 4 = Emergency, Phase 5 = Famine
These are validated by ground surveys → gold standard for drought impact.

**What is HDX?** UN OCHA Humanitarian Data Exchange — free data portal.

**How to use**: Download IPC data for same country × year → 
if SPI event AND high corroboration score AND IPC Phase 3+ → **triple-validated drought**.

Code below downloads FEWS NET IPC data via HDX API:

In [ ]:
# HDX API — no registration required for read access
def get_hdx_ipc(country_iso3='ETH', year=2022):
    """
    Search HDX for IPC/food security datasets for given country.
    Returns dataset metadata list.
    """
    url = 'https://data.humdata.org/api/3/action/package_search'
    params = {
        'q': f'ipc food security {country_iso3} {year}',
        'fq': 'groups:{}'.format(country_iso3.lower()),
        'rows': 5,
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        results = r.json().get('result',{}).get('results',[])
        return [{
            'title':    d.get('title',''),
            'url':      d.get('url',''),
            'notes':    d.get('notes','')[:200],
            'modified': d.get('metadata_modified',''),
        } for d in results]
    except Exception as e:
        print(f"HDX error: {e}")
        return []

# FEWS NET API (text reports, free)
def get_fewsnet_report(country_code='ET', year=2022):
    """
    FEWS NET country codes (2-letter): ET=Ethiopia, KE=Kenya, SO=Somalia, NG=Niger
    Returns situation report URLs.
    """
    url = f'https://fews.net/api/v1/reports/?country_code={country_code}&created={year}'
    try:
        r = requests.get(url, timeout=15)
        reports = r.json().get('results', [])
        return [{
            'title': rp.get('title',''),
            'url':   rp.get('url',''),
            'date':  rp.get('created',''),
        } for rp in reports[:5]]
    except Exception as e:
        print(f"FEWS NET error: {e}")
        return []

# Test
print("HDX IPC data for Ethiopia 2022:")
hdx = get_hdx_ipc('ETH', 2022)
for d in hdx:
    print(f"  {d['title'][:70]}")

print("\nFEWS NET reports for Ethiopia 2022:")
fews = get_fewsnet_report('ET', 2022)
for r in fews:
    print(f"  [{r['date'][:10]}] {r['title'][:70]}")

## Step 13 — Export Results

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

df_regions.to_csv(f'data/nb05_regions_{ts}.csv', index=False)
df_events.to_csv(f'data/nb05_event_scores_{ts}.csv', index=False)
if not df_matched.empty:
    df_matched.to_csv(f'data/nb05_articles_geotagged_{ts}.csv', index=False)

os.makedirs('data', exist_ok=True)
df_regions.to_csv(f'data/nb05_regions_{ts}.csv', index=False)
df_events.to_csv(f'data/nb05_event_scores_{ts}.csv', index=False)
if not df_matched.empty:
    df_matched.to_csv(f'data/nb05_articles_geotagged_{ts}.csv', index=False)

print(f"Saved:")
print(f"  data/nb05_regions_{ts}.csv ({len(df_regions)} regions)")
print(f"  data/nb05_event_scores_{ts}.csv ({len(df_events)} drought events)")
if not df_matched.empty:
    print(f"  data/nb05_articles_geotagged_{ts}.csv ({len(df_matched)} matched articles)")
print()
print("=== Summary ===")
print(f"SPI pixels processed: {len(df_work):,}")
print(f"Regions clustered:    {len(df_regions)}")
print(f"Articles fetched:     {len(df_arts)}")
print(f"Articles geotagged:   {len(df_geo)}")
print(f"Articles matched:     {len(df_matched)}")
print(f"Drought events:       {len(df_events)}")
if len(df_events):
    conf = (df_events['spatial_score'] >= 0.7).sum()
    part = ((df_events['spatial_score'] >= 0.3) & (df_events['spatial_score'] < 0.7)).sum()
    none = (df_events['spatial_score'] < 0.3).sum()
    print(f"  CONFIRMED  (≥0.7): {conf}")
    print(f"  PARTIAL  (0.3-0.7): {part}")
    print(f"  NO EVIDENCE (<0.3): {none}")

## Step 14 — Concept Summary

### What NB05 adds over NB04
```
NB04: country + year → Google News → corroboration score
NB05: pixel (lat,lon) + date window
         → grid cluster → region
         → GDELT 65K sources → article
         → spaCy NER → article lat/lon
         → cKDTree 1° buffer → spatial match
         → spatial_corroboration_score per region
         → weighted aggregate per drought_id
         → geopandas heatmap
```

### Key design decisions
| Decision | Why |
|----------|-----|
| 1° grid cluster (not DBSCAN) | DBSCAN on 60K pts = 12GB RAM. Grid = O(n), memory safe |
| cKDTree spatial match | Fastest for large point sets; <2% error at Africa/Asia latitudes |
| 1° buffer | Covers ERA5 cluster + 50km NER geoparsing error |
| ±30 day temporal lag | Drought onset → media coverage delay |
| spaCy not mordecai3 | mordecai3 = better but needs 500MB GeoNames DB. spaCy = lighter |
| HDX/FEWS NET | X API dead ($5K/mo). HDX = free humanitarian ground truth |

### Three-layer validation
```
Layer 1: SPI signal           (remote sensing, pixel level)
Layer 2: News corroboration   (GDELT, 65K sources, geotagged)
Layer 3: HDX/FEWS NET         (field survey, IPC phase classification)

All three agree → high-confidence drought with real-world impact.
```

### Upgrade path
- Replace spaCy + Nominatim with **mordecai3** for better sub-city geoparsing
- Use **GDELT GKG BigQuery** for pre-geotagged articles (V2Locations field)
- Add ReliefWeb body text as additional article source
- Use **multilingual pipeline from NB04** (XLM-R + Helsinki-NLP) for non-English GDELT articles